In [1]:
# Importing libraries
import os
import itertools
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from mp_api.client import MPRester
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings("ignore")

c:\Users\ahedo\Documents\coding\Pymatgen\pymatgenenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Data harvesting for three categories of HEA (Refractoriness, Corrosion resistant and Lightweight)
import os
from dotenv import load_dotenv
from mp_api.client import MPRester
import warnings
warnings.filterwarnings("ignore")

# Loading environment variables
load_dotenv()
api_key = os.getenv("MY_API_KEY")

#Elements used
elements = ['Al', 'Ti', 'Sc', 'Zr', 'V']
true_vecs = {'Al': 3, 'Ti': 4, 'Sc': 3, 'Zr': 4, 'V': 5}
shear_moduli = {'Al': 26, 'Ti': 44, 'Sc': 29, 'Zr': 33, 'V': 47} 

master_props = {}

print(">>> Starting  HEA Discovery <<<")
print("\n[1/3] Harvesting Materials Data...")

# Data harvesting
with MPRester(api_key) as mpr:
    for el in elements:
        try:
            docs = mpr.materials.summary.search(elements=[el], is_stable=True)
            if docs:
                struct = docs[0].structure
                master_props[el] = {
                    'r': struct.species[0].atomic_radius,
                    'vec': true_vecs[el],
                    'density': struct.density
                }
                print(f"  [+] {el} harvested. (Radius: {master_props[el]['r']:.2f} Å, Density: {master_props[el]['density']:.2f})")
        except Exception as e:
            print(f"  [-] Error fetching {el}: {e}")

# Combinatorial engine for HEA element filtering
print("\n[2/3] Running Combinatorial Engine (5% to 35% fractions)...")
allowed_percentages = range(5, 40, 5)
valid_compositions = []

for combo in itertools.product(allowed_percentages, repeat=len(elements)):
    if sum(combo) == 100:
        comp = dict(zip(elements, [x/100.0 for x in combo]))
        valid_compositions.append(comp)

results = []
for comp in valid_compositions:
    r_avg = sum(frac * master_props[el]['r'] for el, frac in comp.items())
    variance_sum = sum(frac * (1 - master_props[el]['r'] / r_avg)**2 for el, frac in comp.items())
    delta = 100 * np.sqrt(variance_sum)
    
    density = sum(frac * master_props[el]['density'] for el, frac in comp.items())
    results.append({**comp, 'Delta': round(delta, 3), 'Density': round(density, 2)})

df_results = pd.DataFrame(results)

# Filtering using lattice strain (Delta < 6.6) for the  mixed-phase system (HCP)
df_stable = df_results[df_results['Delta'] < 6.6].copy()
print(f"  -> {len(df_results)} theoretical alloys generated.")
print(f"  -> {len(df_stable)} survived the lattice strain filter.")

if df_stable.empty:
    print("FATAL: No alloys survived the physical filter. Adjust your sandbox.")
else:
    # Multi-objective optimisation (Pareto front)
    print("\n[3/3] Training model to discover the optimal Specific Strength...")
    
    # Calculating target strength using Rule of Mixtures
    df_stable['Target_Strength'] = [sum(row[el] * shear_moduli[el] for el in elements) for _, row in df_stable.iterrows()]
    
    X = df_stable[elements]
    model_density = RandomForestRegressor(n_estimators=100, random_state=42).fit(X, df_stable['Density'])
    model_strength = RandomForestRegressor(n_estimators=100, random_state=42).fit(X, df_stable['Target_Strength'])
    
    df_stable['Pred_Density'] = model_density.predict(X)
    df_stable['Pred_Strength'] = model_strength.predict(X)
    
    # Determining strength to weight ratio
    df_stable['Specific_Strength'] = df_stable['Pred_Strength'] / df_stable['Pred_Density']
    
    best_alloy = df_stable.loc[df_stable['Specific_Strength'].idxmax()]
    print("Material discovered")
    
    comp_string = " - ".join([f"{el}:{best_alloy[el]*100:.0f}%" for el in elements])
    print(f"Composition: {comp_string}")
    print(f"Density:     {best_alloy['Pred_Density']:.2f} g/cm³")
    print(f"Strength:    {best_alloy['Pred_Strength']:.2f} GPa")
    print(f"Score:       {best_alloy['Specific_Strength']:.2f} GPa/(g/cm³)")

>>> Starting  HEA Discovery <<<

[1/3] Harvesting Materials Data...


Retrieving SummaryDoc documents: 100%|██████████| 1938/1938 [00:14<00:00, 130.29it/s]


  [+] Al harvested. (Radius: 1.25 Å, Density: 2.72)


Retrieving SummaryDoc documents: 100%|██████████| 990/990 [00:00<00:00, 8288145.63it/s]


  [+] Ti harvested. (Radius: 1.40 Å, Density: 4.67)


...

  [+] Sc harvested. (Radius: 1.60 Å, Density: 2.98)


Retrieving SummaryDoc documents: 100%|██████████| 916/916 [00:00<00:00, 4725685.69it/s]


  [+] Zr harvested. (Radius: 1.55 Å, Density: 6.45)


Retrieving SummaryDoc documents: 100%|██████████| 1016/1016 [00:01<00:00, 930.61it/s]


  [+] V harvested. (Radius: 1.35 Å, Density: 6.38)

[2/3] Running Combinatorial Engine (5% to 35% fractions)...
  -> 1451 theoretical alloys generated.
  -> 4 survived the lattice strain filter.

[3/3] Training model to discover the optimal Specific Strength...
Material discovered
Composition: Al:20% - Ti:35% - Sc:5% - Zr:5% - V:35%
Density:     4.95 g/cm³
Strength:    40.28 GPa
Score:       8.14 GPa/(g/cm³)


In [3]:
# Generating cell
import random
from pymatgen.core import Lattice, Structure, Species
from pymatgen.io.cif import CifWriter
import warnings

warnings.filterwarnings("ignore")

print("\n[4] Creating 3D physical blueprint...")

TOTAL_ATOMS = 54

# Calculating how many whole atoms each element should get
atom_counts = {el: int(round(best_alloy[el] * TOTAL_ATOMS)) for el in elements}

difference = TOTAL_ATOMS - sum(atom_counts.values())
if difference != 0:
    max_el = max(atom_counts, key=atom_counts.get)
    atom_counts[max_el] += difference

print(f"Discrete Atom Mapping for {TOTAL_ATOMS}-atom cell:")
for el, count in atom_counts.items():
    print(f"  {el}: {count} atoms")

# Creating the list of 54 neutral atoms
atom_list = []
for el, count in atom_counts.items():
    atom_list.extend([Species(el, 0)] * count)

# Shuffling the atoms randomly 
random.seed(42)  
random.shuffle(atom_list)

# Building a dummy 1x1x1 BCC unit cell then expand to 3x3x3 (54 empty coordinates)
lattice = Lattice.cubic(3.25)
dummy_struct = Structure(lattice, ["H", "H"], [[0.0, 0.0, 0.0], [0.5, 0.5, 0.5]])
dummy_struct.make_supercell([3, 3, 3])

# Swapping the empty coordinates with the shuffled HEA atoms
for i in range(TOTAL_ATOMS):
    dummy_struct.replace(i, atom_list[i])

# Exporting the final blueprint
file_name = f"Optimal_{''.join(elements)}_Blueprint.cif"
CifWriter(dummy_struct).write_file(file_name)

print("\n>>> Finished <<<")
print(f"Total Atoms in Supercell: {len(dummy_struct)}")
print(f"Saved 3D Blueprint as: '{file_name}'")


[4] Creating 3D physical blueprint...
Discrete Atom Mapping for 54-atom cell:
  Al: 11 atoms
  Ti: 18 atoms
  Sc: 3 atoms
  Zr: 3 atoms
  V: 19 atoms

>>> Finished <<<
Total Atoms in Supercell: 54
Saved 3D Blueprint as: 'Optimal_AlTiScZrV_Blueprint.cif'


In [1]:
from chgnet.model.model import CHGNet
chgnet = CHGNet.load()
from pymatgen.core import Structure
from chgnet.model import StructOptimizer, CHGNet
import warnings

warnings.filterwarnings("ignore")

print("Loading blueprint and ML model...")

# Loading the HEA structure
struct = Structure.from_file("Optimal_AlTiScZrV_Blueprint.cif")

# Loading CHGNet model
chgnet = CHGNet.load()

# Set up optimizer for CPU 
optimizer = StructOptimizer(model=chgnet, use_device="cpu")

# Relaxing the structure
print("Relaxing structure...")
result = optimizer.relax(struct, fmax=0.05)

# Relaxed structure and final energy
relaxed_struct = result["final_structure"]
energy = result["trajectory"].energies[-1]

print("\n>>> Relaxation Complete <<<")
print(f"Final System Energy: {energy:.4f} eV")

# Saving the blueprint
relaxed_struct.to_file("Relaxed_Optimal_AlTiScZrV_Blueprint.cif")
print("Saved relaxed blueprint as: 'Relaxed_Optimal_AlTiScZrV_Blueprint.cif'")

c:\Users\ahedo\Documents\coding\Pymatgen\pymatgenenv\Lib\site-packages\chgnet\graph\converter.py:79: UserWarning: `fast` algorithm is not available, using `legacy`
  warnings.warn(


CHGNet v0.3.0 initialized with 412,525 parameters
CHGNet will run on cpu
Loading blueprint and ML model...
CHGNet v0.3.0 initialized with 412,525 parameters
CHGNet will run on cpu
CHGNet will run on cpu
Relaxing structure...
      Step     Time          Energy          fmax
FIRE:    0 22:19:42     -401.425903        0.711776
FIRE:    1 22:19:43     -401.526581        0.688629
FIRE:    2 22:19:44     -401.712891        0.637902
FIRE:    3 22:19:45     -401.957275        0.569131
FIRE:    4 22:19:45     -402.224640        0.563176
FIRE:    5 22:19:46     -402.481171        0.553989
FIRE:    6 22:19:46     -402.699982        0.542461
FIRE:    7 22:19:47     -402.873840        0.527490
FIRE:    8 22:19:48     -403.013489        0.490906
FIRE:    9 22:19:48     -403.115845        0.442519
FIRE:   10 22:19:49     -403.192627        0.385412
FIRE:   11 22:19:50     -403.267853        0.358552
FIRE:   12 22:19:50     -403.350006        0.352974
FIRE:   13 22:19:51     -403.432068        0.3014

In [ ]:
#  Inverse design and matminer integration
import random
import numpy as np
import pandas as pd
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings("ignore")

print(">>> Initiating<<<\n")

# Defining the sandbox and target properties
elements = ['Al', 'Ti', 'Sc', 'Zr', 'V']
shear_moduli = {'Al': 26, 'Ti': 44, 'Sc': 29, 'Zr': 33, 'V': 47} 
densities = {'Al': 2.72, 'Ti': 4.67, 'Sc': 2.98, 'Zr': 6.45, 'V': 6.38}

# Matminer setup and extraction of quantum features
print("[1/3] Loading Magpie...")
ep_feat = ElementProperty.from_preset(preset_name="magpie")
feature_labels = ep_feat.feature_labels()

# Generating a small synthetic training dataset using Random Combinations
print("[2/3] Training the model on advanced Matminer features...")
training_data = []
for _ in range(500): 
    fractions = np.random.dirichlet(np.ones(len(elements)), size=1)[0]
    comp_dict = {el: frac for el, frac in zip(elements, fractions)}
    
    # Target calculations based on rule of mixtures
    target_density = sum(frac * densities[el] for el, frac in comp_dict.items())
    target_strength = sum(frac * shear_moduli[el] for el, frac in comp_dict.items())
    
    comp = Composition(comp_dict)
    features = ep_feat.featurize(comp) 
    
    training_data.append(features + [target_density, target_strength])

df_train = pd.DataFrame(training_data, columns=feature_labels + ['Target_Density', 'Target_Strength'])

# Training the model on the Matminer features
X = df_train[feature_labels]
ai_density = RandomForestRegressor(n_estimators=50, random_state=42).fit(X, df_train['Target_Density'])
ai_strength = RandomForestRegressor(n_estimators=50, random_state=42).fit(X, df_train['Target_Strength'])

# Using genetic algorithm
print("\n[3/3] Genetic algorithm for alloy formation...")

POPULATION_SIZE = 50
GENERATIONS = 20
MUTATION_RATE = 0.1

def create_random_alloy():
    fractions = np.random.dirichlet(np.ones(len(elements)), size=1)[0]
    return {el: frac for el, frac in zip(elements, fractions)}

def calculate_fitness(alloy_dict):
    comp = Composition(alloy_dict)
    features = ep_feat.featurize(comp)
    
    # Model predicting the properties based on Matminer features
    pred_density = ml_density.predict([features])[0]
    pred_strength = ml_strength.predict([features])[0]
    
    # Specific Strength for the alloys
    return pred_strength / pred_density, pred_density, pred_strength

def crossover(parent1, parent2):
    child = {el: (parent1[el] + parent2[el]) / 2 for el in elements}
    return child

def mutate(alloy):
    el1, el2 = random.sample(elements, 2)
    shift = random.uniform(0.01, 0.05)
    if alloy[el1] > shift:
        alloy[el1] -= shift
        alloy[el2] += shift
    return alloy

# Starting Population
population = [create_random_alloy() for _ in range(POPULATION_SIZE)]
best_alloy_ever = None
best_fitness_ever = 0
best_stats = (0, 0)

for gen in range(GENERATIONS):
    # Scoring the population
    scored_population = []
    for alloy in population:
        fitness, p_dens, p_str = calculate_fitness(alloy)
        scored_population.append((fitness, alloy, p_dens, p_str))
        
        # Tracking the best alloy
        if fitness > best_fitness_ever:
            best_fitness_ever = fitness
            best_alloy_ever = alloy
            best_stats = (p_dens, p_str)
            
    # Sorting by specific strength
    scored_population.sort(key=lambda x: x[0], reverse=True)
    survivors = [x[1] for x in scored_population[:POPULATION_SIZE//2]]
    
    # forming the next generation
    next_generation = []
    while len(next_generation) < POPULATION_SIZE:
        p1, p2 = random.sample(survivors, 2)
        child = crossover(p1, p2)
        if random.random() < MUTATION_RATE:
            child = mutate(child)
        next_generation.append(child)
        
    population = next_generation
    if (gen+1) % 5 == 0:
        print(f"  -> Generation {gen+1} complete. Best Score: {best_fitness_ever:.2f}")

print("Process complete")
comp_string = " - ".join([f"{el}:{best_alloy_ever[el]*100:.1f}%" for el in elements])
print(f"Composition: {comp_string}")
print(f"Density:     {best_stats[0]:.2f} g/cm³")
print(f"Strength:    {best_stats[1]:.2f} GPa")
print(f"Score:       {best_fitness_ever:.2f} GPa/(g/cm³)")

c:\Users\ahedo\Documents\coding\Pymatgen\pymatgenenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


>>> INITIATING INVERSE DESIGN ENGINE <<<

[1/3] Loading Matminer Quantum Features (Magpie)...
[2/3] Training the AI on advanced Matminer features...

[3/3] Unleashing Genetic Algorithm to breed the ultimate alloy...
  -> Generation 5 complete. Best Score: 8.87
  -> Generation 10 complete. Best Score: 8.91
  -> Generation 15 complete. Best Score: 8.99
  -> Generation 20 complete. Best Score: 8.99

🧬 EVOLUTION COMPLETE: OPTIMAL ALLOY DISCOVERED 🧬
Composition: Al:24.4% - Ti:29.4% - Sc:34.4% - Zr:1.3% - V:10.4%
Density:     3.84 g/cm³
Strength:    34.54 GPa
Score:       8.99 GPa/(g/cm³)


In [2]:
import joblib

joblib.dump(ai_density, 'ml_density.model')
joblib.dump(ai_strength, 'ml_strength.model')
print("Exportation complete")

Exportation complete


In [3]:
# =========================================================
# THE UNIVERSAL AI TRAINER (ALL 4 CATEGORIES)
# =========================================================
import joblib
import numpy as np
import pandas as pd
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings("ignore")

# All 17 unique elements across your 4 categories
all_elements = ['Al', 'Ti', 'Sc', 'Zr', 'V', 'Mg', 'Li', 'Zn', 'W', 'Mo', 'Ta', 'Nb', 'Co', 'Cr', 'Fe', 'Ni', 'Cu']

# Master physical properties dictionary
densities = {
    'Al': 2.72, 'Ti': 4.67, 'Sc': 2.98, 'Zr': 6.45, 'V': 6.38,
    'Mg': 1.74, 'Li': 0.53, 'Zn': 7.14, 'W': 19.25, 'Mo': 10.28,
    'Ta': 16.69, 'Nb': 8.57, 'Co': 8.90, 'Cr': 7.19, 'Fe': 7.87,
    'Ni': 8.90, 'Cu': 8.96
}

shear_moduli = {
    'Al': 26, 'Ti': 44, 'Sc': 29, 'Zr': 33, 'V': 47,
    'Mg': 17, 'Li': 4.2, 'Zn': 43, 'W': 161, 'Mo': 126,
    'Ta': 69, 'Nb': 38, 'Co': 82, 'Cr': 115, 'Fe': 82,
    'Ni': 76, 'Cu': 48
}

print("Loading Matminer Magpie descriptors...")
ep_feat = ElementProperty.from_preset("magpie")
feature_labels = ep_feat.feature_labels()

print("Synthesizing 5,000 Universal Training points...")
training_data = []
for _ in range(5000):
    # Select a random subset of 5 elements to mimic a realistic HEA family
    subset = np.random.choice(all_elements, 5, replace=False)
    fractions = np.random.dirichlet(np.ones(5), size=1)[0]
    comp_dict = {el: frac for el, frac in zip(subset, fractions)}
    
    target_density = sum(frac * densities[el] for el, frac in comp_dict.items())
    target_strength = sum(frac * shear_moduli[el] for el, frac in comp_dict.items())
    
    comp = Composition(comp_dict)
    features = ep_feat.featurize(comp)
    
    training_data.append(features + [target_density, target_strength])

df_train = pd.DataFrame(training_data, columns=feature_labels + ['Target_Density', 'Target_Strength'])

print("Training Master ML Models (Using all CPU cores)...")
X = df_train[feature_labels]
# n_jobs=-1 forces Python to use all your laptop's CPU cores for faster training
ml_density = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(X, df_train['Target_Density'])
ml_strength = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(X, df_train['Target_Strength'])

# Save the new master brains directly into your web folder!
# NOTE: Update this path if your MetaForge-Web folder is located somewhere else
joblib.dump(ml_density, 'MetaForge-Web/ml_density.model')
joblib.dump(ml_strength, 'MetaForge-Web/ml_strength.model')

print("\n✅ Universal Models Exported Successfully!")
print("Your Web App is now capable of predicting all 4 HEA categories.")

Loading Matminer Magpie descriptors...
Synthesizing 5,000 Universal Training points...
Training Master ML Models (Using all CPU cores)...

✅ Universal Models Exported Successfully!
Your Web App is now capable of predicting all 4 HEA categories.
